# Prepare corpus

Export a pinned DuYu split when needed, then process its supplied labels with a pinned IDS dictionary and acoustic inventory.


In [4]:
from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'datasets').is_dir() and (candidate / 'src').is_dir():
            return candidate
    raise RuntimeError('Could not find a project root containing datasets/ and src/')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from phonetics.artifacts import sha256_file
from phonetics.source_preparation import export_duyu

DUYU_EXPORT_DIR = PROJECT_ROOT / "datasets/source-data/duyu-hanzi-pinyin"
DUYU_REVISION = "9961d7d05910be7006f2dcfae21f4fc5ce4d560a"
DUYU_SPLIT = "train"
# Source columns for the pinned revision.
DUYU_COLUMNS = {"text": "我们试试看！", "pinyin": "wo3 men shi4 shi4 kan4 ！"}
TONE_CONVENTION = "neutral_unmarked"

RUN_EXPORT = True


In [5]:
if RUN_EXPORT:
    assert "REPLACE_WITH" not in repr((DUYU_EXPORT_DIR, DUYU_REVISION, DUYU_SPLIT, DUYU_COLUMNS))
    exported_path = export_duyu(DUYU_EXPORT_DIR, revision=DUYU_REVISION, split=DUYU_SPLIT,
                               columns=DUYU_COLUMNS, tone_convention=TONE_CONVENTION, allow_download=True)
    print(exported_path)
    print("corpus.csv sha256:", sha256_file(exported_path / "corpus.csv"))
else:
    print("Source export disabled.")


d:\projects\Jupyter\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


D:\projects\Mandarin-Phonetics-Modeling\project\datasets\source-data\duyu-hanzi-pinyin
corpus.csv sha256: 83e95c32329b0395d6561061dcd4b7d61e3cc7d2cdd51938711db6908ee75a2c


## Process the source corpus

Use the existing pinned source CSV. The processor validates the complete dictionary, uses the original character whenever a decomposition contains `？`, excludes whole rows with characters absent from the mapping, and preserves label alignment. It applies pronunciation-conflict, acoustic-coverage, duplicate and explicit sampling rules. Each stage reports rows, unique characters and occurrences. No experiment splits are created.


In [8]:
from phonetics import prepare_corpus

RUN_PROCESSING = True
RAW_CORPUS_PATH = DUYU_EXPORT_DIR / 'corpus.csv'
RAW_CORPUS_SHA256 = '83e95c32329b0395d6561061dcd4b7d61e3cc7d2cdd51938711db6908ee75a2c'
PROCESSED_CORPUS_OUTPUT_ROOT = PROJECT_ROOT / 'datasets/processed-corpus'
CORPUS_ID = "duyu-chinese-lessons-zh-cn-v1"
ACOUSTIC_BUNDLE_PATH = PROJECT_ROOT / 'datasets/pinyin-embeddings/duyu-chinese-lessons-zh-cn-v1'
ACOUSTIC_MANIFEST_SHA256 = "0280ba452aa1580b7d414a2aa15f05d878bd1be2bf11e56f93133d07495e3641"
# Select the sampling parameters before enabling processing.
SAMPLING = {'size': 100_000, 'strategy': 'uniform', 'seed': 42}
IDS_PIN = {
    'path': str(PROJECT_ROOT / 'datasets/ids-decomposition/dictionary.txt'),
    'sha256': '744bb05d5b0742e9ee35c37791f94d56a173349b3367569e7ca11e510364d203',
    'snapshot_commit': '618dbab8a8ddefb958763c8b4afbaa741a4460de',
    'license': 'LGPL-3.0-or-later',
    'attribution': 'Make Me a Hanzi; dictionary.txt derived from Unihan and CJKlib',
}
PROCESSING_CONFIG = {
    'corpus_id': CORPUS_ID,
    'output_root': str(PROCESSED_CORPUS_OUTPUT_ROOT),
    'corpus': {'path': str(RAW_CORPUS_PATH), 'sha256': RAW_CORPUS_SHA256,
               'source_id': 'duyu', 'tone_convention': 'explicit'},
    'ids': IDS_PIN,
    'sampling': SAMPLING,
    'acoustic_bundle': {'path': str(ACOUSTIC_BUNDLE_PATH) if ACOUSTIC_BUNDLE_PATH is not None else None,
                        'manifest_sha256': ACOUSTIC_MANIFEST_SHA256},
}


In [9]:
if RUN_PROCESSING:
    if not all((CORPUS_ID, ACOUSTIC_BUNDLE_PATH, ACOUSTIC_MANIFEST_SHA256)):
        raise ValueError('Select the output ID and pin the acoustic input before processing.')
    processed_corpus_path = prepare_corpus(PROCESSING_CONFIG)
    print(processed_corpus_path)
    print('manifest.json sha256:', sha256_file(processed_corpus_path / 'manifest.json'))
else:
    print('Corpus processing disabled; review inputs, output ID and sampling before enabling it.')


D:\projects\Mandarin-Phonetics-Modeling\project\datasets\processed-corpus\duyu-chinese-lessons-zh-cn-v1
manifest.json sha256: f3395a74ea91ef39e78d068e911194b084e033f5b1075be3dcc490e7abb7b74e
